# Iterate: 실행 → 관찰 → 수정

이 notebook이 시작점입니다. agent가 하는 가장 보편적인 일을 수행하며 Managed Agents API의 인터페이스를 배웁니다. 무언가를 시도하고, 결과를 읽고, 다시 시도합니다. 의도적으로 버그 두 개를 넣은 작은 package를 업로드하고 agent에 테스트를 통과시키도록 지시한 뒤, 루프를 수행하는 모습을 살펴봅니다. 테스트 실행, traceback 읽기, 코드 편집, 재실행을 모두 통과할 때까지 반복합니다.

이 과정에서 나머지 cookbook의 기반이 되는 모든 API 형태를 보게 됩니다. agent / environment / session, 파일 mount, event stream, archive 호출입니다. 이 notebook을 마칠 때면 agent를 처음부터 끝까지 진행하는 데 필요한 모든 작업을 해 본 것입니다.

## 개념

알아야 할 resource는 세 가지입니다.

- **Agent**: 재사용 가능한 설정(model, system prompt, tool)
- **Environment**: container template(package, networking)
- **Session**: agent와 environment를 연결하고, agent에 필요한 파일을 mount하며, event stream을 생성합니다.

agent와 environment는 한 번 생성하여 여러 세션에서 재사용합니다. 각 세션은 하나의 독립적인 실행입니다.


In [ ]:
import os
from pathlib import Path

from anthropic import Anthropic

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")

client = Anthropic()
FIXTURE = Path("example_data") / "iterate"

## 1. agent 생성하기

system prompt는 의도적으로 간략합니다. agent가 단계별 script를 따르는 대신 iterate 루프를 스스로 알아내게 하고 싶기 때문입니다. 테스트 출력만으로도 추가 안내 없이 작업이 충분히 명확해집니다.

`agent_toolset_20260401`은 built-in toolset입니다. bash, read, write, edit, glob, grep, web_fetch, web_search가 포함됩니다. `permission_policy`를 `always_allow`로 설정하면 agent는 확인을 위한 왕복 과정 없이 실행할 수 있습니다.


In [ ]:
agent = client.beta.agents.create(
    name="cookbook-iterate",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-iterate-fix-failing-tests"},
    model=MODEL,
    system=(
        "You are a debugging agent. Your job is to make failing tests pass. "
        "Run the tests, read the failures, fix the code, repeat until green. "
        "Stop when every assertion passes."
    ),
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
        }
    ],
)

## 2. environment 생성하기

environment는 container template입니다. `type: cloud`는 Anthropic이 호스팅하는 sandbox에서 실행됩니다. `networking: limited`는 임의의 외부 traffic을 차단합니다. 이 notebook은 네트워크 접근이 전혀 필요 없으므로 제한된 상태로 유지합니다.


In [ ]:
env = client.beta.environments.create(
    name="cookbook-iterate-env",
    config={"type": "cloud", "networking": {"type": "limited"}},
)

## 3. 실패하는 테스트 업로드하기

Files API로 파일을 업로드하여 ID를 받습니다. 4단계에서 세션에 mount하겠습니다.

`calc.py`에는 의도적으로 넣은 버그 두 개가 있고 `test_calc.py`에는 이를 잡는 assertion 세 개가 있습니다. 실패 중 하나(`test_mean`)는 다른 두 실패의 결과로 발생하며, 이를 통해 agent는 은연중에 과도하게 수정하지 않는 법을 배웁니다. `mean()`은 내부에서 `add`와 `divide`를 호출하므로 이 둘을 수정하면 `mean()`을 직접 편집하지 않아도 `test_mean`이 저절로 통과하기 시작합니다.


In [ ]:
calc_file = client.beta.files.upload(
    file=("calc.py", (FIXTURE / "calc.py").read_bytes(), "text/x-python")
)
test_file = client.beta.files.upload(
    file=("test_calc.py", (FIXTURE / "test_calc.py").read_bytes(), "text/x-python")
)
print(f"uploaded: {calc_file.id}, {test_file.id}")

## 4. 세션 생성하기

세션은 agent와 environment를 연결하고, agent에 필요한 파일을 mount하며, 새 container를 시작합니다. `resources=`는 agent 시작 전에 container에 데이터를 넣는 방법입니다. orchestrate notebook은 같은 필드로 개별 파일을 mount하는 대신 GitHub repo를 clone하는 방법을 보여 줍니다.

파일은 읽기 전용인 `/mnt/session/uploads/<mount_path>` 아래에 mount됩니다. agent는 편집하기 전에 `/mnt/user`나 `/tmp` 같은 쓰기 가능한 디렉터리로 파일을 복사해야 하며, 나중에 가져오려는 것은 `/mnt/session/outputs/`에 둡니다.


In [ ]:
session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=[
        {"type": "file", "file_id": calc_file.id, "mount_path": "calc.py"},
        {"type": "file", "file_id": test_file.id, "mount_path": "test_calc.py"},
    ],
    title="Get the tests green",
)
print(f"session: {session.id}")

## 5. agent를 진행시키고 작업 관찰하기

두 단계입니다. 작업을 담은 `user.message` event를 보내고, agent가 `end_turn`에 도달할 때까지 event stream을 읽습니다.

stream은 server-sent event 연결입니다. agent가 약 30초 동안 반복 작업을 하고 각 회차를 실시간으로 보고 싶으므로 polling 대신 사용합니다(마지막의 보충 설명 참고).

익혀 둘 패턴은 두 가지입니다.

1. **stream을 먼저 열고 전송하세요.** `with` block은 SSE 연결을 엽니다. block 안에서 `send`하는 모든 것은 관찰할 수 있음이 보장됩니다. 먼저 전송하면 race 구간에 발생한 event를 놓칠 위험이 있습니다.
2. **`stop_reason.type == "end_turn"`인 `session.status_idle`에서 종료하세요.** 세션은 입력을 기다릴 때마다 idle이 됩니다. turn이 끝날 때와 custom tool 호출에 응답이 필요할 때 모두 해당합니다. `stop_reason.type`이 이를 구분하며 `end_turn`이 종료 신호입니다. gate notebook은 같은 루프의 `requires_action` 측면을 보여 줍니다.


In [ ]:
with client.beta.sessions.events.stream(session.id) as stream:
    client.beta.sessions.events.send(
        session_id=session.id,
        events=[
            {
                "type": "user.message",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "The tests in /mnt/session/uploads/test_calc.py are "
                            "failing. Copy both files into /mnt/user, iterate "
                            "on calc.py until every test passes, then write the "
                            "final calc.py to /mnt/session/outputs/calc.py. "
                            "pytest isn't installed here, run the assertions "
                            "directly with `python3 -c ...` instead."
                        ),
                    }
                ],
            }
        ],
    )
    print("--- iterate loop ---")
    for ev in stream:
        match ev.type:
            case "agent.message":
                for b in ev.content:
                    if b.type == "text":
                        print(b.text, end="")
            case "agent.tool_use":
                print(f"\n[{ev.name}]")
            case "session.status_idle" if ev.stop_reason and ev.stop_reason.type == "end_turn":
                break
            case "session.status_terminated":
                break

해당 `match ev.type:` block은 표준 streaming 패턴입니다. 이 cookbook의 다른 모든 notebook은 루프를 반복해서 작성하는 대신 `utilities.py`에서 `stream_until_end_turn`으로 가져옵니다. 아래 검증 단계에서도 사용합니다.

`wait_for_idle_status`는 `utilities.py`의 두 번째 helper입니다. 7단계의 안내에서 설명하는 race를 처리합니다. stream이 `session.status_idle`을 내보낸 뒤에도 서버 측 세션 레코드의 `status` 필드는 잠시 `running`으로 읽힐 수 있어, 즉시 `archive()`를 호출하면 400이 발생합니다. helper는 필드가 안정될 때까지 `sessions.retrieve`를 polling할 뿐입니다. streaming 직후 archive하는 코드에는 이 helper가 필요합니다.


In [ ]:
from utilities import stream_until_end_turn, wait_for_idle_status

## 6. 검증하기

agent의 말만 믿지 마세요. 모든 assertion을 독립적으로 한 번 더 실행하고 최종 `calc.py`를 출력합니다. 마지막 루프 내 실행과 turn 종료 사이에 agent가 과도하게 수정하거나 regression을 일으켰다면 이를 잡아냅니다.


In [ ]:
client.beta.sessions.events.send(
    session_id=session.id,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Re-run every assertion from "
                        "/mnt/session/uploads/test_calc.py one more time "
                        "against your final calc.py with `python3 -c ...` "
                        "to confirm they all pass, then cat the final "
                        "/mnt/session/outputs/calc.py."
                    ),
                }
            ],
        }
    ],
)
stream_until_end_turn(client, session.id)

## 7. 정리하기

archive는 세션, environment 또는 agent를 완료된 것으로 표시하는 방법입니다. 실행 중인 container를 종료하고 resource를 workspace quota 계산에서 제외하며 기본 목록에서 숨깁니다. 하지만 audit과 나중에 ID로 resource를 조회하려는 사람을 위해 레코드, 설정, event 이력을 보존합니다. 레코드를 완전히 제거하려면 대부분의 resource가 별도 `delete` endpoint도 제공합니다(operate notebook에서 resource lifecycle을 자세히 설명합니다). 하지만 실행이 끝날 때는 거의 항상 `archive`가 적합합니다.


In [ ]:
wait_for_idle_status(client, session.id)
client.beta.sessions.archive(session.id)
client.beta.environments.archive(env.id)
client.beta.agents.archive(agent.id)
print("archived")

## 보충 설명: streaming 대신 polling하기

5단계의 streaming 패턴은 agent가 몇 초 이상 걸릴 작업의 진행 상황을 실시간으로 보고 싶을 때 적합합니다. 더 짧은 작업이나 장시간 HTTP 연결을 유지하고 싶지 않은 production 코드에서는 대신 `events.list` polling으로 같은 작업을 할 수 있습니다.

```python
client.beta.sessions.events.send(session_id=..., events=[...])
while True:
    time.sleep(2)
    events = client.beta.sessions.events.list(session.id).data
    last = events[-1] if events else None
    if last and last.type == "session.status_terminated":
        break
    if (
        last
        and last.type == "session.status_idle"
        and last.stop_reason
        and last.stop_reason.type == "end_turn"
    ):
        break
# event를 순회하여 agent.message text 출력하기
```

장단점:

agent가 일하는 모습을 보고 싶다면 streaming이 유리합니다. 모든 tool 호출, 모든 부분 message, 모든 상태 전환이 서버에서 발생하는 즉시 도착하며, 이는 새 workflow를 개발할 때 원하는 동작입니다. 대가는 장시간 SSE 연결입니다. process가 turn 내내 살아 있어야 하고, 중간에 멈췄다 재개할 수 없으며, 좋지 않은 순간의 네트워크 끊김은 중단 지점에서 깔끔하게 복구할 방법 없이 stream 도중 연결을 끊을 수 있습니다.

반대 상황에서는 polling이 유리합니다. stateless이고 process 재시작에도 견디며 연결을 계속 열어 두고 싶지 않은 webhook handler 및 queue worker와 자연스럽게 결합됩니다. 대가는 latency와 보이지 않는 진행 상황입니다. 다시 polling하기 전에는 아무것도 보이지 않으므로 feedback은 polling 간격에 제한되며, 긴 turn은 끝날 때까지 조용한 것처럼 보입니다.

agent가 몇 분 동안 실행될 수 있고 handler가 연결을 계속 열어 둘 수 없는 production 설정에서는 polling 패턴(또는 gate notebook에 있는 production용 유사 패턴인 `session.status_idled` webhook)이 적합합니다.


## 다음으로 살펴볼 내용

iterate 루프는 agent 루프의 가장 간단한 형태입니다. 이 디렉터리의 관련 notebook 네 개는 같은 API 형태를 기반으로 진행할 수 있는 다른 workflow를 보여 줍니다.

- [`CMA_orchestrate_issue_to_pr.ipynb`](CMA_orchestrate_issue_to_pr.ipynb) — 긴 tool 체인에서 상태를 유지하는 multi-turn agent입니다. issue를 읽고, 수정하고, PR을 열고, CI 실패에서 복구하고, review comment를 처리하고, merge합니다.
- [`CMA_explore_unfamiliar_codebase.ipynb`](CMA_explore_unfamiliar_codebase.ipynb) — 한 번도 보지 않은 repo에 투입된 agent를 위한 grounding 패턴이며 오래된 문서 함정이 들어 있습니다. 실행 중인 세션에 context를 추가하는 `sessions.resources.add`도 보여 줍니다.
- [`CMA_gate_human_in_the_loop.ipynb`](CMA_gate_human_in_the_loop.ipynb) — human-in-the-loop workflow를 위한 custom tool `decide()` 및 `escalate()` 왕복 과정입니다. `requires_action`에 의한 idle 전환과 병렬 tool 호출 중복 제거를 다룹니다.
- [`CMA_operate_in_production.ipynb`](CMA_operate_in_production.ipynb) — production 설정 설명입니다. vault에 저장된 MCP credential, 장시간 연결 없이 HITL을 수행하는 `session.status_idled` webhook, resource lifecycle의 CRUD 동작을 다룹니다.
